In [0]:
# %run /Workspace/Users/n.moukayed@gmail.com/databricks_projects/Notebooks/NB1_Collibri_Bronze

In [0]:
import logging
import sys
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True,
)

logger = logging.getLogger("bronze_unit_tests")

TEST_TABLE = "interviews_dev.bronze.turbine_raw_unit_test"

In [0]:
def create_test_dataframe(data, schema):
    """Create a Spark DataFrame for a test."""
    return spark.createDataFrame(data, schema)


def drop_test_table():
    """Remove the dedicated unit-test table, if it exists."""
    spark.sql(f"DROP TABLE IF EXISTS {TEST_TABLE}")


def assert_columns_exist(df, expected_columns):
    """Assert all expected columns exist in a DataFrame."""
    missing_columns = set(expected_columns) - set(df.columns)
    assert not missing_columns, (
        f"Missing expected columns: {sorted(missing_columns)}"
    )


def bronze_schema(include_rescued_data=True):
    """Return the schema used by Bronze merge tests."""
    fields = [
        StructField("turbine_id", IntegerType(), False),
        StructField("timestamp", TimestampType(), False),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", IntegerType(), True),
        StructField("power_output", DoubleType(), True),
        StructField("_source_file", StringType(), False),
        StructField("_file_name", StringType(), False),
        StructField("_file_group", StringType(), False),
        StructField("_ingested_at", TimestampType(), False),
        StructField("_ingestion_date", DateType(), False),
    ]

    if include_rescued_data:
        fields.append(StructField("_rescued_data", StringType(), True))

    return StructType(fields)

# Unit Tests

In [0]:
def extract_file_group(df):
    """Extract numeric file group from a data_group_N.csv filename."""
    return df.withColumn(
        "_file_group",
        F.regexp_extract(
            F.col("_file_name"),
            r"data_group_(\d+)\.csv",
            1,
        ),
    )

def deduplicate_data(df):
    """Keep the newest ingested record per turbine and timestamp."""
    dedupe_window = (
        Window
        .partitionBy("turbine_id", "timestamp")
        .orderBy(F.col("_ingested_at").desc())
    )

    return (
        df
        .withColumn("_row_number", F.row_number().over(dedupe_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )


def merge_into_bronze(source_df, target_table):
    """
    Create a Delta target table if it does not exist.
    Otherwise, update matching turbine/timestamp records and insert new records.
    """
    if not spark.catalog.tableExists(target_table):
        (
            source_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
        return "created"

    target_delta = DeltaTable.forName(spark, target_table)

    (
        target_delta.alias("target")
        .merge(
            source_df.alias("source"),
            """
            target.turbine_id = source.turbine_id
            AND target.timestamp = source.timestamp
            """,
        )
        .whenMatchedUpdate(set={
            "wind_speed": "source.wind_speed",
            "wind_direction": "source.wind_direction",
            "power_output": "source.power_output",
            "_rescued_data": "source._rescued_data",
            "_source_file": "source._source_file",
            "_file_name": "source._file_name",
            "_file_group": "source._file_group",
            "_ingested_at": "source._ingested_at",
            "_ingestion_date": "source._ingestion_date",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

    return "merged"


def calculate_data_quality_metrics(target_table):
    """Calculate basic quality metrics for a Bronze table."""
    return spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT turbine_id) AS unique_turbines,
            MIN(timestamp) AS min_timestamp,
            MAX(timestamp) AS max_timestamp
        FROM {target_table}
    """)


def get_sample_data(target_table):
    """Return a sorted sample from a Bronze table."""
    return (
        spark.table(target_table)
        .orderBy("_ingestion_date", "turbine_id", "timestamp")
        .limit(10)
    )

def test_file_group_extraction():
    """Validate file group extraction for valid and invalid filenames."""
    logger.info("Running test_file_group_extraction")

    input_df = spark.createDataFrame(
        [
            ("data_group_1.csv", "1"),
            ("data_group_2.csv", "2"),
            ("data_group_99.csv", "99"),
            ("unexpected_file.csv", ""),
            ("data_group_invalid.csv", ""),
            ("data_group_1.txt", ""),
        ],
        ["_file_name", "expected_file_group"],
    )

    result_df = extract_file_group(input_df)

    failed_rows = (
        result_df
        .filter(F.col("_file_group") != F.col("expected_file_group"))
        .count()
    )

    assert failed_rows == 0, (
        f"File-group extraction failed for {failed_rows} filename(s)."
    )

    logger.info("test_file_group_extraction passed")

def test_deduplicate_data_removes_duplicates():
    """Validate duplicate keys are reduced to the latest ingested record."""
    logger.info("Running test_deduplicate_data_removes_duplicates")

    schema = StructType([
        StructField("turbine_id", IntegerType(), False),
        StructField("timestamp", TimestampType(), False),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", IntegerType(), True),
        StructField("power_output", DoubleType(), True),
        StructField("_ingested_at", TimestampType(), False),
        StructField("_source_file", StringType(), False),
    ])

    test_df = create_test_dataframe(
        [
            (
                1,
                datetime(2022, 3, 1, 0, 0),
                9.1,
                269,
                2.9,
                datetime(2026, 8, 19, 10, 0),
                "file1.csv",
            ),
            (
                1,
                datetime(2022, 3, 1, 0, 0),
                9.1,
                269,
                3.1,
                datetime(2026, 8, 19, 11, 0),
                "file1.csv",
            ),
            (
                1,
                datetime(2022, 3, 1, 1, 0),
                9.2,
                270,
                3.0,
                datetime(2026, 8, 19, 10, 0),
                "file1.csv",
            ),
            (
                2,
                datetime(2022, 3, 1, 0, 0),
                11.3,
                316,
                2.5,
                datetime(2026, 8, 19, 10, 0),
                "file2.csv",
            ),
        ],
        schema,
    )

    result_df = deduplicate_data(test_df)

    assert result_df.count() == 3, "Expected one duplicate record to be removed."

    retained_row = (
        result_df
        .filter(
            (F.col("turbine_id") == 1)
            & (F.col("timestamp") == datetime(2022, 3, 1, 0, 0))
        )
        .first()
    )

    assert retained_row["power_output"] == 3.1, (
        "Expected newest ingested duplicate record to be retained."
    )

    logger.info("test_deduplicate_data_removes_duplicates passed")


def test_deduplicate_data_keeps_unique_records():
    """Validate unique records are not removed."""
    logger.info("Running test_deduplicate_data_keeps_unique_records")

    schema = StructType([
        StructField("turbine_id", IntegerType(), False),
        StructField("timestamp", TimestampType(), False),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", IntegerType(), True),
        StructField("power_output", DoubleType(), True),
        StructField("_ingested_at", TimestampType(), False),
    ])

    test_df = create_test_dataframe(
        [
            (
                1,
                datetime(2022, 3, 1, 0, 0),
                9.1,
                269,
                2.9,
                datetime(2026, 8, 19, 10, 0),
            ),
            (
                1,
                datetime(2022, 3, 1, 1, 0),
                9.2,
                270,
                3.0,
                datetime(2026, 8, 19, 10, 0),
            ),
            (
                2,
                datetime(2022, 3, 1, 0, 0),
                11.3,
                316,
                2.5,
                datetime(2026, 8, 19, 10, 0),
            ),
        ],
        schema,
    )

    result_df = deduplicate_data(test_df)

    assert result_df.count() == 3, "All unique records should be retained."

    logger.info("test_deduplicate_data_keeps_unique_records passed")

def test_merge_into_bronze_creates_table():
    """Validate merge creates a table when the target does not exist."""
    logger.info("Running test_merge_into_bronze_creates_table")

    drop_test_table()

    try:
        test_df = create_test_dataframe(
            [
                (
                    1,
                    datetime(2022, 3, 1, 0, 0),
                    9.1,
                    269,
                    2.9,
                    "file1.csv",
                    "file1.csv",
                    "1",
                    datetime(2026, 8, 19, 10, 0),
                    datetime(2026, 8, 19).date(),
                    None,
                ),
            ],
            bronze_schema(),
        )

        result = merge_into_bronze(test_df, TEST_TABLE)

        assert result == "created", "Expected table creation status."
        assert spark.catalog.tableExists(TEST_TABLE), "Expected test table to exist."
        assert spark.table(TEST_TABLE).count() == 1, "Expected one row in test table."

        logger.info("test_merge_into_bronze_creates_table passed")

    finally:
        drop_test_table()


def test_merge_into_bronze_inserts_new_records():
    """Validate merge inserts a source key not already present in the target."""
    logger.info("Running test_merge_into_bronze_inserts_new_records")

    drop_test_table()

    try:
        initial_df = create_test_dataframe(
            [
                (
                    1,
                    datetime(2022, 3, 1, 0, 0),
                    9.1,
                    269,
                    2.9,
                    "file1.csv",
                    "file1.csv",
                    "1",
                    datetime(2026, 8, 19, 10, 0),
                    datetime(2026, 8, 19).date(),
                    None,
                ),
            ],
            bronze_schema(),
        )

        new_df = create_test_dataframe(
            [
                (
                    1,
                    datetime(2022, 3, 1, 0, 0),
                    9.1,
                    269,
                    2.9,
                    "file1.csv",
                    "file1.csv",
                    "1",
                    datetime(2026, 8, 19, 10, 0),
                    datetime(2026, 8, 19).date(),
                    None,
                ),
                (
                    2,
                    datetime(2022, 3, 1, 0, 0),
                    11.3,
                    316,
                    2.5,
                    "file2.csv",
                    "file2.csv",
                    "2",
                    datetime(2026, 8, 19, 10, 0),
                    datetime(2026, 8, 19).date(),
                    None,
                ),
            ],
            bronze_schema(),
        )

        assert merge_into_bronze(initial_df, TEST_TABLE) == "created"
        assert merge_into_bronze(new_df, TEST_TABLE) == "merged"

        result_df = spark.table(TEST_TABLE)

        assert result_df.count() == 2, "Expected the new turbine record to be inserted."
        assert (
            result_df.filter(F.col("turbine_id") == 2).count() == 1
        ), "Expected Turbine 2 record to exist."

        logger.info("test_merge_into_bronze_inserts_new_records passed")

    finally:
        drop_test_table()


def test_merge_into_bronze_updates_existing_records():
    """Validate merge updates a matching turbine/timestamp record."""
    logger.info("Running test_merge_into_bronze_updates_existing_records")

    drop_test_table()

    try:
        initial_df = create_test_dataframe(
            [
                (
                    1,
                    datetime(2022, 3, 1, 0, 0),
                    9.1,
                    269,
                    2.9,
                    "file1.csv",
                    "file1.csv",
                    "1",
                    datetime(2026, 8, 19, 10, 0),
                    datetime(2026, 8, 19).date(),
                    None,
                ),
            ],
            bronze_schema(),
        )

        updated_df = create_test_dataframe(
            [
                (
                    1,
                    datetime(2022, 3, 1, 0, 0),
                    12.0,
                    300,
                    4.1,
                    "file1_updated.csv",
                    "file1_updated.csv",
                    "1",
                    datetime(2026, 8, 19, 11, 0),
                    datetime(2026, 8, 19).date(),
                    None,
                ),
            ],
            bronze_schema(),
        )

        assert merge_into_bronze(initial_df, TEST_TABLE) == "created"
        assert merge_into_bronze(updated_df, TEST_TABLE) == "merged"

        result_df = spark.table(TEST_TABLE)

        assert result_df.count() == 1, "Expected matched record to be updated, not inserted."

        updated_row = result_df.first()

        assert updated_row["power_output"] == 4.1, "Expected power output to update."
        assert updated_row["wind_speed"] == 12.0, "Expected wind speed to update."
        assert updated_row["wind_direction"] == 300, (
            "Expected wind direction to update."
        )
        assert updated_row["_file_name"] == "file1_updated.csv", (
            "Expected metadata to update."
        )

        logger.info("test_merge_into_bronze_updates_existing_records passed")

    finally:
        drop_test_table()


def test_calculate_data_quality_metrics():
    """Validate expected quality metrics are returned for a known test table."""
    logger.info("Running test_calculate_data_quality_metrics")

    drop_test_table()

    try:
        test_df = create_test_dataframe(
            [
                (
                    1,
                    datetime(2022, 3, 1, 0, 0),
                    9.1,
                    269,
                    2.9,
                    "file1.csv",
                    "file1.csv",
                    "1",
                    datetime(2026, 8, 19, 10, 0),
                    datetime(2026, 8, 19).date(),
                    None,
                ),
                (
                    2,
                    datetime(2022, 3, 2, 0, 0),
                    11.3,
                    316,
                    2.5,
                    "file2.csv",
                    "file2.csv",
                    "2",
                    datetime(2026, 8, 19, 10, 0),
                    datetime(2026, 8, 19).date(),
                    None,
                ),
            ],
            bronze_schema(),
        )

        (
            test_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(TEST_TABLE)
        )

        metrics = calculate_data_quality_metrics(TEST_TABLE).first()

        assert metrics["total_rows"] == 2
        assert metrics["unique_turbines"] == 2
        assert metrics["min_timestamp"] == datetime(2022, 3, 1, 0, 0)
        assert metrics["max_timestamp"] == datetime(2022, 3, 2, 0, 0)

        logger.info("test_calculate_data_quality_metrics passed")

    finally:
        drop_test_table()


def test_get_sample_data():
    """Validate sample retrieval returns expected records."""
    logger.info("Running test_get_sample_data")

    drop_test_table()

    try:
        test_df = create_test_dataframe(
            [
                (
                    2,
                    datetime(2022, 3, 1, 1, 0),
                    11.3,
                    316,
                    2.5,
                    "file2.csv",
                    "file2.csv",
                    "2",
                    datetime(2026, 8, 19, 10, 0),
                    datetime(2026, 8, 20).date(),
                    None,
                ),
                (
                    1,
                    datetime(2022, 3, 1, 0, 0),
                    9.1,
                    269,
                    2.9,
                    "file1.csv",
                    "file1.csv",
                    "1",
                    datetime(2026, 8, 19, 10, 0),
                    datetime(2026, 8, 19).date(),
                    None,
                ),
            ],
            bronze_schema(),
        )

        (
            test_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(TEST_TABLE)
        )

        sample_df = get_sample_data(TEST_TABLE)

        assert sample_df.count() == 2
        assert_columns_exist(
            sample_df,
            [
                "turbine_id",
                "timestamp",
                "wind_speed",
                "wind_direction",
                "power_output",
                "_ingestion_date",
            ],
        )

        first_row = sample_df.first()

        assert first_row["turbine_id"] == 1, (
            "Expected sample rows to be sorted by ingestion date and turbine ID."
        )

        logger.info("test_get_sample_data passed")

    finally:
        drop_test_table()


# Execute Unit Tests

In [0]:

def run_all_tests():
    """Execute all Bronze unit tests and fail the notebook if any test fails."""
    logger.info("=" * 80)
    logger.info("Starting Unit Tests for Bronze Layer")
    logger.info("=" * 80)

    tests = [
        test_file_group_extraction,
        test_deduplicate_data_removes_duplicates,
        test_deduplicate_data_keeps_unique_records,
        test_merge_into_bronze_creates_table,
        test_merge_into_bronze_inserts_new_records,
        test_merge_into_bronze_updates_existing_records,
        test_calculate_data_quality_metrics,
        test_get_sample_data,
    ]

    passed = 0
    failed = 0

    for test_func in tests:
        try:
            test_func()
            passed += 1
        except Exception as exc:
            failed += 1
            logger.error(
                "%s failed: %s",
                test_func.__name__,
                exc,
                exc_info=True,
            )

    logger.info("Test Results: %s passed, %s failed", passed, failed)

    if failed > 0:
        raise AssertionError(f"{failed} test(s) failed")

    logger.info("All tests passed!")


run_all_tests()